# 1. Data Preparation

## 1.1. Data Loading

1.1.1. Install and import all required Python libraries

In [1]:
!pip install pandas

In [2]:
import pandas as pd
import os
from google.colab import drive

1.1.2. Mount Google Drive to read the dataset (excluding data dictionary)

In [3]:
drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
base = "/content/drive/MyDrive/North Star Case Study Dataset"

app_events = pd.read_csv(f"{base}/app_events.csv")
complaints = pd.read_csv(f"{base}/complaints.csv")
customers = pd.read_csv(f"{base}/customers.csv")
deliveries = pd.read_csv(f"{base}/deliveries.csv")
drivers = pd.read_csv(f"{base}/drivers.csv")
hubs = pd.read_csv(f"{base}/hubs.csv")
incidents = pd.read_csv(f"{base}/incidents.csv")
orders = pd.read_csv(f"{base}/orders.csv")
vehicles = pd.read_csv(f"{base}/vehicles.csv")

## 1.2. Zone Standardization

Make sure all zone variations in its respective column - across all CSVs - are stripped of whitespace, made lowercase and mapped to a discrete set of possible zone options (i.e. standardization)

In [5]:
ZONE_MAP = {
    "ctr": "Central",
    "central": "Central",
    "north": "North",
    "south": "South",
    "east": "East",
    "west": "West",
    "airport": "Airport",
    "riverside": "Riverside",
}


def clean_zone(series):
    return series.str.strip().str.lower().map(ZONE_MAP)


customers["home_zone"] = clean_zone(customers["home_zone"])
orders["pickup_zone"] = clean_zone(orders["pickup_zone"])
orders["dropoff_zone"] = clean_zone(orders["dropoff_zone"])
drivers["base_zone"] = clean_zone(drivers["base_zone"])
vehicles["assigned_zone"] = clean_zone(vehicles["assigned_zone"])
hubs["zone"] = clean_zone(hubs["zone"])
app_events["zone_context"] = clean_zone(app_events["zone_context"])

## 1.3. Timestamp Parsing

Convert all datetime columns into a Pandas datetime object. If a value cannot be parsed, they are converted to `NaT` (Not a Time) rather than being dropped

In [6]:
def parse_timestamp(df, cols):
    for col in cols:
        df[col] = pd.to_datetime(df[col], errors="coerce")


parse_timestamp(customers, ["signup_date"])
parse_timestamp(orders, ["order_created_at"])
parse_timestamp(deliveries, ["dispatch_time", "delivery_completed_at"])
parse_timestamp(complaints, ["created_at"])
parse_timestamp(incidents, ["reported_at"])
parse_timestamp(app_events, ["event_timestamp"])

## 1.4. Type Coercion

Cast any columns that are flags into integer, and force numeric columns that may have been imported as an object.

In [7]:
flag_cols = {
    "orders": ["special_handling_flag"],
    "deliveries": ["proof_of_completion_missing"],
    "drivers": ["active_flag"],
    "app_events": ["success_flag"],
}

frame_map = {
    "orders": orders,
    "deliveries": deliveries,
    "drivers": drivers,
    "app_events": app_events,
}

for name, cols in flag_cols.items():
    for col in cols:
        frame_map[name][col] = pd.to_numeric(
            frame_map[name][col], errors="coerce"
        ).astype("Int64")

numeric_cols = {
    "customers": ["loyalty_score", "app_engagement_score"],
    "drivers": ["training_score", "driver_rating", "years_experience"],
    "vehicles": ["battery_health_pct", "odometer_km"],
    "orders": ["order_value", "promised_window_hours"],
    "deliveries": [
        "route_distance_km",
        "manual_route_override_count",
        "customer_rating_post_delivery",
        "fuel_or_charge_cost",
    ],
    "incidents": ["resolved_hours"],
    "complaints": ["resolution_days", "compensation_amount"],
    "app_events": ["api_latency_ms"],
}

frame_map_full = {
    "customers": customers,
    "drivers": drivers,
    "vehicles": vehicles,
    "orders": orders,
    "deliveries": deliveries,
    "incidents": incidents,
    "complaints": complaints,
    "app_events": app_events,
}

for name, cols in numeric_cols.items():
    for col in cols:
        frame_map_full[name][col] = pd.to_numeric(
            frame_map_full[name][col], errors="coerce"
        )

## 1.5. Blank Handling

Fill any known categorical blanks with "Unknown" or zero (aka. sentinel values) rather than dropping rows.

In [8]:
orders["booking_channel"] = (
    orders["booking_channel"].replace("", "Unknown").fillna("Unknown")
)
customers["preferred_channel"] = (
    customers["preferred_channel"].replace("", "Unknown").fillna("Unknown")
)
complaints["compensation_amount"] = complaints["compensation_amount"].fillna(0.0)

## 1.6. Export

In [9]:
# Make a dedicated directory in the Google Drive folder for the "cleaned" dataset
out = f"{base}/cleaned"
os.makedirs(out, exist_ok=True)

exports = {
    "hubs_clean.csv": hubs,
    "customers_clean.csv": customers,
    "drivers_clean.csv": drivers,
    "vehicles_clean.csv": vehicles,
    "orders_clean.csv": orders,
    "deliveries_clean.csv": deliveries,
    "incidents_clean.csv": incidents,
    "complaints_clean.csv": complaints,
    "app_events_clean.csv": app_events,
}

for filename, df in exports.items():
    df.to_csv(f"{out}/{filename}", index=False)
    print(f"{filename}: {len(df)} rows")

hubs_clean.csv: 8 rows
customers_clean.csv: 650 rows
drivers_clean.csv: 170 rows
vehicles_clean.csv: 120 rows
orders_clean.csv: 1250 rows
deliveries_clean.csv: 950 rows
incidents_clean.csv: 280 rows
complaints_clean.csv: 320 rows
app_events_clean.csv: 640 rows
